# SupportHub Ticket Dataset — Exploratory Data Analysis

This notebook analyses the training dataset and records the decisions that shape the model architecture.

**Data sources:**
- `data/processed/ticket_sample.csv` — 226 human-labeled tickets (ground truth)
- `data/processed/train.csv` — augmented dataset after paraphrase expansion (used if it exists)

Run cells top to bottom. Decision cells are marked **DECISION**.

In [ ]:
import os
import sys
import csv
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Paths — run from repo root or notebooks/ directory
BASE = os.path.join(os.path.dirname(os.path.abspath('.')), 'SupportHub-ml') \
    if 'notebooks' not in os.getcwd() else os.path.dirname(os.getcwd())
BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))

SAMPLE_PATH = os.path.join(BASE, 'data', 'processed', 'ticket_sample.csv')
TRAIN_PATH  = os.path.join(BASE, 'data', 'processed', 'train.csv')

LABEL_ORDER  = ['critical', 'high', 'medium', 'low']
LABEL_COLORS = {'critical': '#ef4444', 'high': '#f97316', 'medium': '#eab308', 'low': '#22c55e'}

print('BASE:', BASE)
print('ticket_sample exists:', os.path.exists(SAMPLE_PATH))
print('train.csv exists    :', os.path.exists(TRAIN_PATH))

---
## 1. Load Data

In [ ]:
def load_csv(path):
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    if lines and not lines[0].strip():
        lines = lines[1:]
    return pd.DataFrame(list(csv.DictReader(lines)))

# Ground truth — always load
df_source = load_csv(SAMPLE_PATH)
df_source['human_label'] = df_source['human_label'].str.strip().str.lower()
df_source['suggested_label'] = df_source['suggested_label'].str.strip().str.lower()
print(f'Source (ground truth): {len(df_source)} rows')
df_source.head(3)

In [ ]:
# Augmented training set — load if it exists, otherwise fall back to source
if os.path.exists(TRAIN_PATH):
    df_train = pd.read_csv(TRAIN_PATH)
    df_train['label'] = df_train['label'].str.strip().str.lower()
    print(f'Augmented train.csv: {len(df_train)} rows')
    augmented = True
else:
    df_train = df_source.rename(columns={'human_label': 'label'})[['title', 'description', 'label']].copy()
    df_train['source_type'] = 'original'
    print('train.csv not found — using source only (run augment_dataset.py first)')
    augmented = False

df_train.head(3)

---
## 2. Label Distribution

Compare the raw distribution of `human_label` (ground truth) vs `suggested_label` (LLM-only) in the source, and show how the augmented set preserves that distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(
    axes,
    ['human_label', 'suggested_label'],
    ['Human Label (ground truth)', 'Suggested Label (LLM-only)']
):
    counts = df_source[col].value_counts().reindex(LABEL_ORDER, fill_value=0)
    bars = ax.bar(counts.index, counts.values,
                  color=[LABEL_COLORS[l] for l in counts.index])
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 1, str(v),
                ha='center', va='bottom', fontsize=11)

plt.suptitle('Source Dataset Label Distribution (n=226)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Human label counts:', df_source['human_label'].value_counts().to_dict())
agree = (df_source['human_label'] == df_source['suggested_label']).mean()
print(f'Human–AI agreement: {agree:.1%}')

In [ ]:
if augmented:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Full augmented distribution
    counts_all = df_train['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    bars = axes[0].bar(counts_all.index, counts_all.values,
                       color=[LABEL_COLORS[l] for l in counts_all.index])
    axes[0].set_title('Augmented train.csv (all)', fontsize=13)
    axes[0].set_ylabel('Count')
    for bar, v in zip(bars, counts_all.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, v + 1, str(v),
                     ha='center', va='bottom', fontsize=11)

    # Side-by-side original vs paraphrase
    orig_counts = df_train[df_train['source_type'] == 'original']['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    para_counts = df_train[df_train['source_type'] == 'paraphrase']['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    x = np.arange(len(LABEL_ORDER))
    w = 0.35
    axes[1].bar(x - w/2, orig_counts.values, w, label='original',
                color=[LABEL_COLORS[l] for l in LABEL_ORDER], alpha=0.9)
    axes[1].bar(x + w/2, para_counts.values, w, label='paraphrase',
                color=[LABEL_COLORS[l] for l in LABEL_ORDER], alpha=0.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(LABEL_ORDER)
    axes[1].set_title('Original vs Paraphrase by label', fontsize=13)
    axes[1].legend()

    plt.suptitle(f'Augmented Dataset (n={len(df_train)})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    print('Augmented label counts:', df_train['label'].value_counts().to_dict())

---
## 3. Human–AI Disagreement Analysis

Understanding where the LLM got the label wrong shapes our expectations for model accuracy and validates using `human_label` as ground truth.

In [ ]:
rank = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df_source['rank_human']    = df_source['human_label'].map(rank)
df_source['rank_suggested'] = df_source['suggested_label'].map(rank)
df_source['delta'] = df_source['rank_human'] - df_source['rank_suggested']

upgrades   = (df_source['delta'] > 0).sum()
downgrades = (df_source['delta'] < 0).sum()
agrees     = (df_source['delta'] == 0).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Agreement pie
axes[0].pie(
    [agrees, upgrades, downgrades],
    labels=[f'Agree ({agrees})', f'Human upgraded ({upgrades})', f'Human downgraded ({downgrades})'],
    colors=['#22c55e', '#ef4444', '#3b82f6'],
    autopct='%1.0f%%', startangle=90
)
axes[0].set_title('Human vs LLM Agreement', fontsize=13)

# Confusion matrix heatmap
conf = pd.crosstab(
    df_source['human_label'].str.capitalize(),
    df_source['suggested_label'].str.capitalize(),
    rownames=['Human'], colnames=['Suggested']
).reindex(index=[l.capitalize() for l in LABEL_ORDER],
          columns=[l.capitalize() for l in LABEL_ORDER],
          fill_value=0)
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues', ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion: Human (row) vs Suggested (col)', fontsize=13)

plt.tight_layout()
plt.show()

print('Key disagreement patterns:')
disagree = df_source[df_source['delta'] != 0][['title', 'suggested_label', 'human_label', 'delta']]
disagree = disagree.sort_values('delta')
print(disagree[['title', 'suggested_label', 'human_label']].to_string(index=False, max_colwidth=55))

---
## 4. Text Length Analysis

Checks whether ticket text is long enough to produce meaningful TF-IDF features.

In [ ]:
df_train['title_len'] = df_train['title'].str.len()
df_train['desc_len']  = df_train['description'].str.len()
df_train['total_len'] = df_train['title_len'] * 2 + df_train['desc_len']  # title doubled in preprocessing

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(
    axes,
    ['title_len', 'desc_len', 'total_len'],
    ['Title length (chars)', 'Description length (chars)', 'Effective length (title×2 + desc)']
):
    for lbl in LABEL_ORDER:
        subset = df_train[df_train['label'] == lbl][col]
        ax.hist(subset, bins=20, alpha=0.6, label=lbl, color=LABEL_COLORS[lbl])
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Characters')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(df_train.groupby('label')[['title_len', 'desc_len', 'total_len']].agg(['mean', 'min', 'max']).round(0))

---
## 5. TF-IDF Keyword Analysis

Confirms that discriminative vocabulary exists between label classes — a prerequisite for TF-IDF to be effective.

In [ ]:
import re

STOP_WORDS = {
    'a','an','the','is','it','in','on','at','to','for','of','and','or','but','not',
    'with','this','that','are','was','were','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might','shall','can',
    'i','we','you','he','she','they','me','us','him','her','them',
    'my','our','your','his','its','their','what','which','who','when','where','why',
    'how','all','each','every','both','few','more','most','other','some','such',
    'no','so','yet','as','if','then','than','too','very','just','also','into',
    'from','up','out','over','again','here','there','any','only','please','resolve',
    'asap','fix','issue','problem','need','help','able'
}

def clean(text):
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df_train['text'] = df_train.apply(
    lambda r: clean(f"{r['title']} {r['title']} {r.get('description', '')}"), axis=1
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
top_n = 15

for ax, lbl in zip(axes.flatten(), LABEL_ORDER):
    corpus = df_train[df_train['label'] == lbl]['text'].tolist()
    others = df_train[df_train['label'] != lbl]['text'].tolist()

    if not corpus:
        ax.set_title(f'{lbl.upper()} — no data')
        continue

    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
    vec.fit(corpus + others)
    X_class = vec.transform(corpus).toarray()
    X_other = vec.transform(others).toarray()

    # Score = mean TF-IDF in class minus mean in others
    score = X_class.mean(axis=0) - X_other.mean(axis=0)
    terms = vec.get_feature_names_out()
    top_idx = score.argsort()[-top_n:][::-1]

    top_terms = [terms[i] for i in top_idx]
    top_scores = [score[i] for i in top_idx]

    ax.barh(top_terms[::-1], top_scores[::-1], color=LABEL_COLORS[lbl], alpha=0.85)
    ax.set_title(f'{lbl.upper()} — top discriminative terms', fontsize=12)
    ax.set_xlabel('Mean TF-IDF (class) − Mean TF-IDF (others)')

plt.suptitle('Most Discriminative Terms per Label Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Class Imbalance Assessment

Quantifies imbalance and determines whether `class_weight='balanced'` is needed.

In [ ]:
counts = df_train['label'].value_counts().reindex(LABEL_ORDER)
total  = len(df_train)
majority = counts.max()

print('Label counts and imbalance ratio:')
for lbl in LABEL_ORDER:
    c = counts[lbl]
    ratio = majority / c if c > 0 else float('inf')
    bar = '█' * int(c / total * 40)
    print(f"  {lbl:8s}  {c:4d}  ({c/total:5.1%})  {bar}  ratio to majority: {ratio:.1f}x")

max_ratio = majority / counts.min()
print(f'\nMax imbalance ratio: {max_ratio:.1f}x')
if max_ratio > 1.5:
    print('→ Imbalance is significant. Use class_weight="balanced" in LogisticRegression.')
else:
    print('→ Imbalance is mild. class_weight="balanced" is still recommended for robustness.')

---
## 7. Model Architecture Decision

**DECISION — Why we use a single classifier instead of the original 3-model setup**

The original architecture had:
- Ridge regression → `sentimentScore`
- RandomForest regression → `complexityScore`  
- LogisticRegression → `priority` + `confidence`

This required `sentimentScore` and `complexityScore` as labelled targets. `ticket_sample.csv` has no such columns — only `human_label`. Training regression models without targets is impossible.

**New approach:** Train a single `LogisticRegression(class_weight='balanced')` classifier on `human_label`. Derive all component scores from the classifier's `predict_proba()` output:

```
P = [P(critical), P(high), P(medium), P(low)]

sentimentScore  = P(critical) + P(high)                         # urgency proxy
complexityScore = 0.8×P(critical) + 0.5×P(high) + 0.2×P(medium)  # difficulty proxy  
aiPriorityScore = 1.0×P(critical) + 0.75×P(high) + 0.25×P(medium) + 0.0×P(low)
confidence      = max(P)                                        # unchanged
agingScore      = deterministic (hours elapsed / 72h cap)       # unchanged
```

This is more honest — the scores come from evidence, not from a separate model trained on LLM-generated labels.

In [ ]:
# Demonstrate the scoring formula on a small example
examples = [
    {'label': 'critical', 'P': [0.82, 0.12, 0.04, 0.02]},
    {'label': 'high',     'P': [0.10, 0.70, 0.15, 0.05]},
    {'label': 'medium',   'P': [0.05, 0.20, 0.65, 0.10]},
    {'label': 'low',      'P': [0.02, 0.05, 0.10, 0.83]},
]

print(f"{'Label':8s}  {'sentiment':9s}  {'complexity':10s}  {'aiScore':8s}  {'confidence':10s}")
print('-' * 58)
for ex in examples:
    pc, ph, pm, pl = ex['P']
    sentiment  = pc + ph
    complexity = 0.8*pc + 0.5*ph + 0.2*pm
    ai_score   = 1.0*pc + 0.75*ph + 0.25*pm + 0.0*pl
    confidence = max(ex['P'])
    print(f"{ex['label']:8s}  {sentiment:.3f}      {complexity:.3f}       {ai_score:.3f}     {confidence:.3f}")

---
## 8. Conclusions & Decisions for Phase 2

| Decision | Value |
|---|---|
| Training source | `train.csv` (augmented) or `ticket_sample.csv` fallback |
| Label column | `human_label` → normalised to lowercase |
| Model | `LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs')` |
| Vectorizer | `TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, sublinear_tf=True)` |
| Saved artifacts | `vectorizer.pkl`, `priority_model.pkl` only (no regression models) |
| Score derivation | From `predict_proba()` using coefficients above |
| Aging | Deterministic, unchanged |
| Class imbalance | `class_weight='balanced'` handles it |
| Test split | 80/20, `stratify=label` |